In [2]:
import json
import os
from pathlib import Path
from collections import defaultdict

# Directory containing the output files
output_dir = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_appropriateness\output")

# Get all JSON files in the output directory
json_files = list(output_dir.glob("cot_*.json"))

print(f"Found {len(json_files)} files in the output directory")
print("\nFiles:")
for f in json_files:
    print(f"  - {f.name}")

Found 30 files in the output directory

Files:
  - cot_meta-llama_Llama-3.1-8B-Instruct_baseline_num_gen_turn_0_by_extracted_from_social.json
  - cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_1_by_meta-llama_Llama-3.1-8B-Instruct.json
  - cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_1_by_mistralai_Ministral-8B-Instruct-2410.json
  - cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_1_by_Qwen_Qwen2.5-7B-Instruct.json
  - cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_2_by_meta-llama_Llama-3.1-8B-Instruct.json
  - cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_2_by_mistralai_Ministral-8B-Instruct-2410.json
  - cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_2_by_Qwen_Qwen2.5-7B-Instruct.json
  - cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_3_by_meta-llama_Llama-3.1-8B-Instruct.json
  - cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_3_by_mistralai_Ministral-8B-Instruct-2410.json
  - co

In [3]:
# Load all files and extract data
file_data = {}

for file_path in json_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Create a dictionary mapping unique_id to the example
    examples_dict = {}
    for example in data:
        if 'unique_id' in example:
            unique_id = example['unique_id']
            examples_dict[unique_id] = example
    
    file_data[file_path.name] = examples_dict
    print(f"Loaded {file_path.name}: {len(examples_dict)} examples")

print(f"\nTotal files loaded: {len(file_data)}")

Loaded cot_meta-llama_Llama-3.1-8B-Instruct_baseline_num_gen_turn_0_by_extracted_from_social.json: 15 examples
Loaded cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_1_by_meta-llama_Llama-3.1-8B-Instruct.json: 15 examples
Loaded cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_1_by_mistralai_Ministral-8B-Instruct-2410.json: 15 examples
Loaded cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_1_by_Qwen_Qwen2.5-7B-Instruct.json: 13 examples
Loaded cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_2_by_meta-llama_Llama-3.1-8B-Instruct.json: 15 examples
Loaded cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_2_by_mistralai_Ministral-8B-Instruct-2410.json: 15 examples
Loaded cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_2_by_Qwen_Qwen2.5-7B-Instruct.json: 14 examples
Loaded cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_3_by_meta-llama_Llama-3.1-8B-Instruct.json: 15 examples
Loaded cot_meta-llama_Llama-3.1-8B-I

In [4]:
# Find common unique_ids across all files
all_unique_ids = [set(examples.keys()) for examples in file_data.values()]
common_unique_ids = set.intersection(*all_unique_ids) if all_unique_ids else set()

print(f"Common examples across all files: {len(common_unique_ids)}")
print(f"\nExample of common unique_ids (first 5): {list(common_unique_ids)[:5]}")

Common examples across all files: 10

Example of common unique_ids (first 5): ['82428958697384800_11336088894941144024', '22344244419519199_17402967697845139214', '6831492794824847_6953515965526014329', '37268771336504051_2585354592208575908', '18442403337591841_10279341821823853675']


In [5]:
# Calculate accuracy for each file
results = {}

for file_name, examples_dict in file_data.items():
    correct = 0
    total = 0
    errors = []
    
    for unique_id in common_unique_ids:
        example = examples_dict[unique_id]
        
        # Get gold label
        gold_label = None
        if 'annotations' in example and example['annotations'] is not None:
            if 'irony_appropriateness_label' in example['annotations']:
                gold_label = example['annotations']['irony_appropriateness_label']
        
        # Get predicted label
        predicted_label = None
        if 'model_answer' in example and example['model_answer'] is not None:
            try:
                predicted_label = int(example['model_answer'])
            except (ValueError, TypeError):
                errors.append({
                    'unique_id': unique_id,
                    'model_answer': example['model_answer'],
                    'reason': 'Invalid model_answer format'
                })
                continue
        
        # Skip if either label is missing
        if gold_label is None or predicted_label is None:
            continue
        
        total += 1
        if gold_label == predicted_label:
            correct += 1
    
    accuracy = correct / total if total > 0 else 0
    results[file_name] = {
        'correct': correct,
        'total': total,
        'accuracy': accuracy,
        'errors': len(errors)
    }

print("Accuracy Results (on common examples only):")
print("=" * 100)
for file_name, result in sorted(results.items()):
    print(f"\n{file_name}")
    print(f"  Correct: {result['correct']}/{result['total']}")
    print(f"  Accuracy: {result['accuracy']:.4f} ({result['accuracy']*100:.2f}%)")
    print(f"  Errors (invalid model_answer): {result['errors']}")

Accuracy Results (on common examples only):

cot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_num_gen_turn_1_by_Qwen_Qwen2.5-7B-Instruct.json
  Correct: 4/10
  Accuracy: 0.4000 (40.00%)
  Errors (invalid model_answer): 0

cot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_num_gen_turn_1_by_meta-llama_Llama-3.1-8B-Instruct.json
  Correct: 6/10
  Accuracy: 0.6000 (60.00%)
  Errors (invalid model_answer): 0

cot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_num_gen_turn_1_by_mistralai_Ministral-8B-Instruct-2410.json
  Correct: 6/10
  Accuracy: 0.6000 (60.00%)
  Errors (invalid model_answer): 0

cot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_num_gen_turn_2_by_Qwen_Qwen2.5-7B-Instruct.json
  Correct: 4/10
  Accuracy: 0.4000 (40.00%)
  Errors (invalid model_answer): 0

cot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_num_gen_turn_2_by_meta-llama_Llama-3.1-8B-Instruct.json
  Correct: 7/10
  Accuracy: 0.7000 (70.00%)
  Errors (invalid model_answer): 0

cot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_num_gen_turn_2_by_mistralai_Ministral-8B-Instruct-241

In [6]:
# Create a summary table
import pandas as pd

summary_data = []
for file_name, result in sorted(results.items()):
    summary_data.append({
        'File': file_name,
        'Correct': result['correct'],
        'Total': result['total'],
        'Accuracy': result['accuracy'],
        'Accuracy (%)': result['accuracy'] * 100,
        'Errors': result['errors']
    })

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values('Accuracy', ascending=False)

print("\nSummary Table (sorted by accuracy):")
print(df_summary.to_string(index=False))


Summary Table (sorted by accuracy):
                                                                                                          File  Correct  Total  Accuracy  Accuracy (%)  Errors
            cot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_num_gen_turn_2_by_mistralai_Ministral-8B-Instruct-2410.json        8     10       0.8          80.0       0
            cot_mistralai_Ministral-8B-Instruct-2410_MultiPICo_num_gen_turn_1_by_Qwen_Qwen2.5-7B-Instruct.json        8     10       0.8          80.0       0
cot_mistralai_Ministral-8B-Instruct-2410_MultiPICo_num_gen_turn_3_by_mistralai_Ministral-8B-Instruct-2410.json        8     10       0.8          80.0       0
                cot_meta-llama_Llama-3.1-8B-Instruct_MultiPICo_num_gen_turn_3_by_Qwen_Qwen2.5-7B-Instruct.json        7     10       0.7          70.0       0
                cot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_num_gen_turn_3_by_meta-llama_Llama-3.1-8B-Instruct.json        7     10       0.7          70.0       0
         